# Arbitrary-N multiscale directional-change representation

This notebook converts the intuition developed from two through five thresholds into one causal representation that works for any strictly ordered threshold grid

$$\Theta=(\theta_1 < \theta_2 < \cdots < \theta_N).$$

It does **not** classify regimes and it does **not** predict propagation. It defines the information a later classifier is allowed to see. Future propagation, rejection, terminal depth, and terminal outcome are deliberately absent from the live feature tables.

In [1]:
import gc
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from dc_basics.core import dc_live_features, parse_dc_events

DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "GBPUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
THRESHOLDS = (0.002, 0.003, 0.0045, 0.00675, 0.010125)
FIRING_WINDOW_BARS = 240
CASE_WINDOW_BARS = 720
PLOT_COLORS = ("#16a34a", "#2563eb", "#7c3aed", "#c026d3", "#0891b2")

In [2]:
from dc_basics.enums import DcMode


class HierarchyShape(StrEnum):
    BROAD_AGREEMENT = "broad_agreement"
    COHERENT_FRONTIER = "coherent_frontier"
    FRAGMENTED = "fragmented"


class LineageResolution(StrEnum):
    REJECTED = "rejected_at_frontier"
    REACHED_OUTER = "reached_outer"
    AMBIGUOUS = "ambiguous_same_bar"
    UNRESOLVED = "unresolved_at_sample_end"


MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}
MODE_SIGN = {DcMode.UP.value: 1, DcMode.DOWN.value: -1}

if len(THRESHOLDS) < 2:
    raise ValueError("At least two thresholds are required.")
if not all(np.isfinite(theta) and theta > 0.0 for theta in THRESHOLDS):
    raise ValueError("Thresholds must be finite and strictly positive.")
if not all(left < right for left, right in zip(THRESHOLDS[:-1], THRESHOLDS[1:], strict=True)):
    raise ValueError("Thresholds must be strictly increasing.")

## Representation and causal contract

The canonical representation has four layers:

1. `scale_state`: one node per `(information_time, threshold_index)`;
2. `edge_state`: one adjacent relationship per `(information_time, lower_threshold_index)`;
3. `global_state`: one whole-hierarchy snapshot per initialized bar;
4. `lineage_state`: optional live context for a broad-agreement small seed.

`lineage_resolutions` is deliberately separate. It is diagnostic and only becomes known on its recorded resolution timestamp. Same-bar lineages are marked ambiguous instead of being assigned a fabricated intrabar order.

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no close column.")

prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Symbol: {SYMBOL}")
print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Symbol: GBPUSD.r
Bars: 258,573
Range: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


## Layer 1: causal per-threshold scale state

`live_tmv`, `live_t`, and `live_r` describe the unfinished current leg. The `completed_*` columns describe the previous completed leg and are forward-filled only from its DCC information timestamp. `reversal_progress` measures the fraction of the next opposite DCC threshold traversed from the causally maintained running extreme.

In [4]:
def build_threshold_catalog(thresholds: tuple[float, ...]) -> pd.DataFrame:
    records = []
    for index, theta in enumerate(thresholds):
        lower_theta = thresholds[index - 1] if index > 0 else np.nan
        records.append({
            "threshold_index": index,
            "threshold_id": f"T{index + 1:02d}",
            "threshold": theta,
            "ratio_to_lower": theta / lower_theta if index > 0 else np.nan,
        })
    return pd.DataFrame.from_records(records).set_index("threshold_index", drop=False)


from dc_basics.events import (
    prepare_threshold_event_ledger as prepare_event_ledger,
)


def build_scale_state(
    price_series: pd.Series, thresholds: tuple[float, ...], firing_window: int,
) -> tuple[pd.DataFrame, dict[int, pd.DataFrame]]:
    catalog = build_threshold_catalog(thresholds)
    positions = np.arange(len(price_series), dtype=np.int64)
    scale_parts = []
    ledgers = {}

    for threshold_index, contract in catalog.iterrows():
        theta = float(contract["threshold"])
        threshold_id = str(contract["threshold_id"])
        live = dc_live_features(price_series, theta=theta)
        ledger = prepare_event_ledger(parse_dc_events(price_series, theta=theta), threshold_id, threshold_index, theta)
        ledgers[threshold_index] = ledger

        mode = live["direction"].map(MODE_MAP)
        mode_known = mode.notna()
        upward_dcc = live["upward_dcc"].astype(bool)
        downward_dcc = live["downward_dcc"].astype(bool)
        dcc_flag = upward_dcc | downward_dcc
        mode_start = mode_known & mode.ne(mode.shift())
        mode_run_id = mode_start.cumsum()
        mode_start_position = pd.Series(np.where(mode_start, positions, np.nan), index=price_series.index).ffill()
        mode_age = pd.Series(positions - mode_start_position, index=price_series.index).where(mode_known)

        running_high = price_series.groupby(mode_run_id).cummax()
        running_low = price_series.groupby(mode_run_id).cummin()
        running_extreme = pd.Series(
            np.where(live["direction"].eq(1.0), running_high, running_low),
            index=price_series.index,
            dtype=np.float64,
        ).where(mode_known)
        upward_progress = (price_series - running_extreme) / running_extreme / theta
        downward_progress = (running_extreme - price_series) / running_extreme / theta
        reversal_progress = pd.Series(
            np.where(live["direction"].eq(1.0), downward_progress, upward_progress),
            index=price_series.index,
            dtype=np.float64,
        ).where(mode_known)
        max_live_tmv = live["tmv_current"].groupby(mode_run_id).cummax().where(mode_known)

        event_lookup = ledger.set_index("dcc_idx")
        dcc_event_id = price_series.index.to_series().map(event_lookup["event_id"])
        current_event_id = dcc_event_id.ffill()
        initialization_mask = mode_known & current_event_id.isna()
        current_event_id = current_event_id.mask(initialization_mask, f"{threshold_id}-INIT")
        dcc_information_time = price_series.index.to_series().where(dcc_flag)
        current_event_information_time = dcc_information_time.ffill()
        if mode_known.any():
            initialization_time = mode_known.index[mode_known.argmax()]
            current_event_information_time = current_event_information_time.mask(
                initialization_mask, initialization_time
            )

        completed_direction = price_series.index.to_series().map(event_lookup["direction"]).ffill()
        completed_tmv = price_series.index.to_series().map(event_lookup["tmv"]).ffill()
        completed_t = price_series.index.to_series().map(event_lookup["t"]).ffill()
        completed_r = price_series.index.to_series().map(event_lookup["r"]).ffill()
        completed_information_time = price_series.index.to_series().map(event_lookup["information_time"]).ffill()
        dcc_count_window = dcc_flag.astype(np.int64).rolling(firing_window, min_periods=1).sum()

        part = pd.DataFrame({
            "time": price_series.index,
            "feature_information_time": price_series.index,
            "bar_position": positions,
            "threshold_index": threshold_index,
            "threshold_id": threshold_id,
            "threshold": theta,
            "ratio_to_lower": contract["ratio_to_lower"],
            "price": price_series.to_numpy(dtype=np.float64),
            "mode": mode.to_numpy(),
            "mode_sign": live["direction"].to_numpy(),
            "mode_age_bars": mode_age.to_numpy(),
            "upward_dcc": upward_dcc.to_numpy(),
            "downward_dcc": downward_dcc.to_numpy(),
            "dcc_flag": dcc_flag.to_numpy(),
            "dcc_event_id": dcc_event_id.to_numpy(),
            "current_event_id": current_event_id.to_numpy(),
            "current_event_information_time": current_event_information_time.to_numpy(),
            "completed_direction": completed_direction.to_numpy(),
            "completed_tmv": completed_tmv.to_numpy(),
            "completed_t": completed_t.to_numpy(),
            "completed_r": completed_r.to_numpy(),
            "completed_event_information_time": completed_information_time.to_numpy(),
            "live_tmv": live["tmv_current"].to_numpy(),
            "live_t": live["t_current"].to_numpy(),
            "live_r": live["rdc_current"].to_numpy(),
            "max_live_tmv": max_live_tmv.to_numpy(),
            "running_extreme": running_extreme.to_numpy(),
            "reversal_progress": reversal_progress.to_numpy(),
            "dcc_count_window": dcc_count_window.to_numpy(),
            "dcc_rate_per_bar": (dcc_count_window / firing_window).to_numpy(),
        })
        scale_parts.append(part)

    return pd.concat(scale_parts, ignore_index=True), ledgers

In [5]:
threshold_catalog = build_threshold_catalog(THRESHOLDS)
scale_state, event_ledgers = build_scale_state(prices, THRESHOLDS, FIRING_WINDOW_BARS)

display(threshold_catalog.style.format({"threshold": "{:.4%}", "ratio_to_lower": "{:.3f}"}))
print(f"Scale-state rows: {len(scale_state):,}")
print(f"Expected rows: {len(prices) * len(THRESHOLDS):,}")

,threshold_index,threshold_id,threshold,ratio_to_lower
threshold_index,,,,
0,0,T01,0.2000%,nan
1,1,T02,0.3000%,1.500
2,2,T03,0.4500%,1.500
3,3,T04,0.6750%,1.500
4,4,T05,1.0125%,1.500


Scale-state rows: 1,292,865
Expected rows: 1,292,865


## Layers 2 and 3: adjacent edges and the whole hierarchy

An edge is active whenever two adjacent modes disagree. Its target direction is the lower-scale mode and its `target_progress` is the upper scale's live reversal progress. The global layer retains the complete mode vector and adds topology-preserving summaries. Aggregates never replace the mode vector.

In [6]:
_SCALE_MATRIX_CACHE: dict[tuple[int, str], pd.DataFrame] = {}


def scale_matrix(scale_frame: pd.DataFrame, column: str, threshold_count: int) -> pd.DataFrame:
    cache_key = (id(scale_frame), column)
    if cache_key not in _SCALE_MATRIX_CACHE:
        matrix = scale_frame.set_index(["time", "threshold_index"])[column].unstack("threshold_index")
        _SCALE_MATRIX_CACHE[cache_key] = matrix.sort_index()
    matrix = _SCALE_MATRIX_CACHE[cache_key]
    if threshold_count > matrix.shape[1]:
        raise ValueError("threshold_count exceeds the cached scale matrix.")
    return matrix.iloc[:, :threshold_count]


def configuration_lookup(threshold_count: int) -> dict[int, str]:
    lookup = {}
    for code in range(2 ** threshold_count):
        modes = [DcMode.UP.value if code & (1 << index) else DcMode.DOWN.value for index in range(threshold_count)]
        lookup[code] = "/".join(modes)
    return lookup


def build_global_state(price_series: pd.Series, scale_frame: pd.DataFrame, threshold_count: int) -> pd.DataFrame:
    mode_sign = scale_matrix(scale_frame, "mode_sign", threshold_count)
    known = mode_sign.notna().all(axis=1)
    mode_sign = mode_sign.loc[known].astype(np.int8)
    times = mode_sign.index
    signs = mode_sign.to_numpy(dtype=np.int8)
    boundaries = signs[:, :-1] != signs[:, 1:]
    boundary_count = boundaries.sum(axis=1).astype(np.int64)
    first_boundary = np.argmax(boundaries, axis=1) + 1
    small_breadth = np.where(boundary_count > 0, first_boundary, threshold_count)
    reversed_boundaries = boundaries[:, ::-1]
    outer_breadth = np.where(boundary_count > 0, np.argmax(reversed_boundaries, axis=1) + 1, threshold_count)
    outer_mode_count = (signs == signs[:, -1, None]).sum(axis=1)

    current_run = np.ones(len(times), dtype=np.int64)
    longest_run = np.ones(len(times), dtype=np.int64)
    for index in range(1, threshold_count):
        current_run = np.where(signs[:, index] == signs[:, index - 1], current_run + 1, 1)
        longest_run = np.maximum(longest_run, current_run)

    weights = (1 << np.arange(threshold_count, dtype=np.int64))
    configuration_code = ((signs > 0).astype(np.int64) * weights).sum(axis=1)
    labels = configuration_lookup(threshold_count)
    configuration = pd.Series(configuration_code, index=times).map(labels)
    shape = np.select(
        [boundary_count == 0, boundary_count == 1],
        [HierarchyShape.BROAD_AGREEMENT.value, HierarchyShape.COHERENT_FRONTIER.value],
        default=HierarchyShape.FRAGMENTED.value,
    )
    counter_depth = np.where(boundary_count == 0, 0.0, np.where(boundary_count == 1, small_breadth, np.nan))

    dcc = scale_matrix(scale_frame, "dcc_flag", threshold_count).loc[times].astype(bool)
    ages = scale_matrix(scale_frame, "mode_age_bars", threshold_count).loc[times].astype(float)
    progress = scale_matrix(scale_frame, "reversal_progress", threshold_count).loc[times].astype(float)
    firing_counts = scale_matrix(scale_frame, "dcc_count_window", threshold_count).loc[times].astype(float)
    firing_rates = scale_matrix(scale_frame, "dcc_rate_per_bar", threshold_count).loc[times].astype(float)
    dcc_count = dcc.sum(axis=1).to_numpy(dtype=np.int64)
    frame = pd.DataFrame({
        "time": times,
        "feature_information_time": times,
        "bar_position": price_series.index.get_indexer(times),
        "price": price_series.loc[times].to_numpy(dtype=np.float64),
        "threshold_count": threshold_count,
        "configuration_code": configuration_code,
        "configuration": configuration.to_numpy(),
        "hierarchy_shape": shape,
        "broad_agreement": boundary_count == 0,
        "coherent_frontier": boundary_count <= 1,
        "direction_boundaries": boundary_count,
        "directional_runs": boundary_count + 1,
        "fragmentation": boundary_count / (threshold_count - 1),
        "small_contiguous_breadth": small_breadth,
        "outer_contiguous_breadth": outer_breadth,
        "outer_mode_count": outer_mode_count,
        "longest_directional_run": longest_run,
        "counter_scale_depth": counter_depth,
        "signed_mode_balance": signs.mean(axis=1),
        "mean_mode_age_bars": ages.mean(axis=1).to_numpy(),
        "max_mode_age_bars": ages.max(axis=1).to_numpy(),
        "mean_reversal_progress": progress.mean(axis=1).to_numpy(),
        "max_reversal_progress": progress.max(axis=1).to_numpy(),
        "dcc_count_this_bar": dcc_count,
        "union_event": dcc_count > 0,
        "multi_scale_same_bar": dcc_count > 1,
        "hierarchy_dcc_count_window": firing_counts.sum(axis=1).to_numpy(),
        "hierarchy_dcc_rate": firing_rates.mean(axis=1).to_numpy(),
    }).set_index("time", drop=False)
    for index in range(threshold_count):
        frame[f"mode_{index:02d}"] = np.where(signs[:, index] > 0, DcMode.UP.value, DcMode.DOWN.value)
    return frame


def build_edge_state(scale_frame: pd.DataFrame, global_frame: pd.DataFrame, threshold_count: int) -> pd.DataFrame:
    times = global_frame.index
    matrices = {
        column: scale_matrix(scale_frame, column, threshold_count).loc[times]
        for column in [
            "mode", "mode_age_bars", "dcc_flag", "dcc_event_id", "current_event_id",
            "live_tmv", "live_t", "live_r", "reversal_progress", "dcc_count_window",
        ]
    }
    threshold_values = (
        scale_frame[["threshold_index", "threshold"]]
        .drop_duplicates("threshold_index")
        .set_index("threshold_index")["threshold"]
        .sort_index()
    )
    parts = []
    bar_positions = global_frame["bar_position"].to_numpy(dtype=np.int64)

    for lower_index in range(threshold_count - 1):
        upper_index = lower_index + 1
        lower_mode = matrices["mode"][lower_index]
        upper_mode = matrices["mode"][upper_index]
        active = lower_mode.ne(upper_mode)
        contest_start = active & ~active.shift(fill_value=False)
        contest_sequence = contest_start.cumsum().astype(np.int64)
        start_positions = pd.Series(np.where(contest_start, bar_positions, np.nan), index=times).ffill()
        contest_age = pd.Series(bar_positions - start_positions, index=times).where(active)
        start_times = pd.Series(times.where(contest_start), index=times).ffill().where(active)
        seed_ids = matrices["current_event_id"][lower_index].where(contest_start).ffill().where(active)
        target_progress = matrices["reversal_progress"][upper_index].where(active)
        max_target_progress = target_progress.groupby(contest_sequence).cummax().where(active)
        lower_dcc = matrices["dcc_flag"][lower_index].astype(bool)
        upper_dcc = matrices["dcc_flag"][upper_index].astype(bool)
        contest_id = pd.Series(
            [f"E{lower_index + 1:02d}-C{number:06d}" for number in contest_sequence],
            index=times,
        ).where(active)

        part = pd.DataFrame({
            "time": times,
            "feature_information_time": times,
            "bar_position": bar_positions,
            "lower_threshold_index": lower_index,
            "upper_threshold_index": upper_index,
            "lower_threshold": threshold_values.loc[lower_index],
            "upper_threshold": threshold_values.loc[upper_index],
            "threshold_ratio": threshold_values.loc[upper_index] / threshold_values.loc[lower_index],
            "lower_mode": lower_mode.to_numpy(),
            "upper_mode": upper_mode.to_numpy(),
            "agreement": (~active).to_numpy(),
            "active_contest": active.to_numpy(),
            "contest_id": contest_id.to_numpy(),
            "contest_seed_event_id": seed_ids.to_numpy(),
            "contest_start_time": start_times.to_numpy(),
            "contest_age_bars": contest_age.to_numpy(),
            "target_mode": lower_mode.where(active).to_numpy(),
            "target_progress": target_progress.to_numpy(),
            "max_target_progress": max_target_progress.to_numpy(),
            "progress_drawdown": (max_target_progress - target_progress).to_numpy(),
            "lower_mode_age_bars": matrices["mode_age_bars"][lower_index].to_numpy(),
            "upper_mode_age_bars": matrices["mode_age_bars"][upper_index].to_numpy(),
            "mode_age_difference": (
                matrices["mode_age_bars"][lower_index] - matrices["mode_age_bars"][upper_index]
            ).to_numpy(),
            "lower_live_tmv": matrices["live_tmv"][lower_index].to_numpy(),
            "upper_live_tmv": matrices["live_tmv"][upper_index].to_numpy(),
            "lower_live_t": matrices["live_t"][lower_index].to_numpy(),
            "upper_live_t": matrices["live_t"][upper_index].to_numpy(),
            "lower_live_r": matrices["live_r"][lower_index].to_numpy(),
            "upper_live_r": matrices["live_r"][upper_index].to_numpy(),
            "lower_dcc_count_window": matrices["dcc_count_window"][lower_index].to_numpy(),
            "upper_dcc_count_window": matrices["dcc_count_window"][upper_index].to_numpy(),
            "lower_dcc_this_bar": lower_dcc.to_numpy(),
            "upper_dcc_this_bar": upper_dcc.to_numpy(),
            "same_bar_transition": (lower_dcc & upper_dcc).to_numpy(),
        })
        parts.append(part)

    return pd.concat(parts, ignore_index=True)

## Layer 4: optional causal broad-root lineage context

A lineage starts only when the previous bar had broad agreement and a small DCC establishes the opposite target. `attained_depth` is updated only after an exact upper-scale DCC. The resolving rejection or outer-reaching bar is excluded from `lineage_state`; it appears only in the separate diagnostic resolution table. A propagated bar may simultaneously become elapsed bar zero of the next stage because that hand-off is known at the close.

In [7]:
def build_causal_lineage_context(
    global_frame: pd.DataFrame, scale_frame: pd.DataFrame, threshold_count: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    times = global_frame.index
    modes = scale_matrix(scale_frame, "mode_sign", threshold_count).loc[times].to_numpy(dtype=np.int8)
    dcc = scale_matrix(scale_frame, "dcc_flag", threshold_count).loc[times].to_numpy(dtype=bool)
    event_ids = scale_matrix(scale_frame, "dcc_event_id", threshold_count).loc[times].to_numpy(dtype=object)
    bar_positions = global_frame["bar_position"].to_numpy(dtype=np.int64)
    live_records = []
    resolution_records = []
    active = None

    def resolve(lineage: dict[str, object], row_index: int, resolution: LineageResolution) -> None:
        resolution_records.append({
            "lineage_id": lineage["lineage_id"],
            "seed_time": lineage["seed_time"],
            "target_mode": lineage["target_mode"],
            "attained_depth": lineage["attained_depth"],
            "resolution": resolution.value,
            "resolution_information_time": times[row_index],
            "resolution_bar_position": bar_positions[row_index],
            "diagnostic_only": True,
        })

    for row_index in range(1, len(times)):
        resolved_on_bar = False
        if active is not None:
            lower_index = int(active["attained_depth"])
            upper_index = lower_index + 1
            target_sign = int(active["target_sign"])
            propagated = dcc[row_index, upper_index] and modes[row_index, upper_index] == target_sign
            rejected = dcc[row_index, lower_index] and modes[row_index, lower_index] != target_sign

            if propagated or rejected:
                resolved_on_bar = True
                zero_duration = bar_positions[row_index] == active["stage_start_position"]
                if (propagated and rejected) or zero_duration:
                    resolve(active, row_index, LineageResolution.AMBIGUOUS)
                    active = None
                elif rejected:
                    resolve(active, row_index, LineageResolution.REJECTED)
                    active = None
                else:
                    active["attained_depth"] = upper_index
                    if upper_index == threshold_count - 1:
                        resolve(active, row_index, LineageResolution.REACHED_OUTER)
                        active = None
                    else:
                        next_upper_index = upper_index + 1
                        next_same_bar = (
                            dcc[row_index, next_upper_index]
                            and modes[row_index, next_upper_index] == target_sign
                        )
                        if next_same_bar:
                            resolve(active, row_index, LineageResolution.AMBIGUOUS)
                            active = None
                        else:
                            active["stage_start_position"] = bar_positions[row_index]
                            active["stage_start_time"] = times[row_index]
                            active["stage_start_event_id"] = event_ids[row_index, upper_index]

        if active is None and not resolved_on_bar:
            prior_broad_agreement = np.all(modes[row_index - 1] == modes[row_index - 1, 0])
            small_seed = dcc[row_index, 0] and modes[row_index, 0] != modes[row_index - 1, 0]
            if prior_broad_agreement and small_seed:
                active = {
                    "lineage_id": event_ids[row_index, 0],
                    "seed_time": times[row_index],
                    "seed_position": bar_positions[row_index],
                    "target_sign": int(modes[row_index, 0]),
                    "target_mode": DcMode.UP.value if modes[row_index, 0] > 0 else DcMode.DOWN.value,
                    "attained_depth": 0,
                    "stage_start_position": bar_positions[row_index],
                    "stage_start_time": times[row_index],
                    "stage_start_event_id": event_ids[row_index, 0],
                }
                medium_same_bar = dcc[row_index, 1] and modes[row_index, 1] == active["target_sign"]
                if medium_same_bar:
                    resolve(active, row_index, LineageResolution.AMBIGUOUS)
                    active = None

        if active is not None:
            attained_depth = int(active["attained_depth"])
            live_records.append({
                "lineage_id": active["lineage_id"],
                "seed_time": active["seed_time"],
                "stage_start_event_id": active["stage_start_event_id"],
                "stage_start_time": active["stage_start_time"],
                "time": times[row_index],
                "feature_information_time": times[row_index],
                "bar_position": bar_positions[row_index],
                "target_mode": active["target_mode"],
                "attained_depth": attained_depth,
                "active_lower_threshold_index": attained_depth,
                "active_upper_threshold_index": attained_depth + 1,
                "lineage_age_bars": bar_positions[row_index] - active["seed_position"],
                "stage_age_bars": bar_positions[row_index] - active["stage_start_position"],
                "timing_clean": True,
            })

    if active is not None:
        resolution_records.append({
            "lineage_id": active["lineage_id"],
            "seed_time": active["seed_time"],
            "target_mode": active["target_mode"],
            "attained_depth": active["attained_depth"],
            "resolution": LineageResolution.UNRESOLVED.value,
            "resolution_information_time": pd.NaT,
            "resolution_bar_position": pd.NA,
            "diagnostic_only": True,
        })

    return pd.DataFrame.from_records(live_records), pd.DataFrame.from_records(resolution_records)

In [8]:
def build_representation_view(
    price_series: pd.Series, scale_frame: pd.DataFrame, threshold_count: int,
) -> dict[str, pd.DataFrame]:
    if not 2 <= threshold_count <= scale_frame["threshold_index"].nunique():
        raise ValueError("threshold_count is outside the available scale-state range.")
    global_frame = build_global_state(price_series, scale_frame, threshold_count)
    edge_frame = build_edge_state(scale_frame, global_frame, threshold_count)
    lineage_frame, resolution_frame = build_causal_lineage_context(global_frame, scale_frame, threshold_count)
    event_frame = global_frame.loc[global_frame["union_event"]].copy()
    return {
        "global_state": global_frame,
        "edge_state": edge_frame,
        "event_state": event_frame,
        "lineage_state": lineage_frame,
        "lineage_resolutions": resolution_frame,
    }

## Arbitrary-N identity and provenance audit

The same builders are now exercised for the first 2, 3, 4, and 5 thresholds. This is an implementation audit, not an economic comparison between grids. Every view uses the same underlying causal node state.

In [9]:
def audit_representation(
    scale_frame: pd.DataFrame, view: dict[str, pd.DataFrame], threshold_count: int,
) -> pd.DataFrame:
    global_frame = view["global_state"]
    edge_frame = view["edge_state"]
    lineage_frame = view["lineage_state"]
    resolution_frame = view["lineage_resolutions"]
    selected_scale = scale_frame.loc[scale_frame["threshold_index"].lt(threshold_count)]

    mode_columns = [f"mode_{index:02d}" for index in range(threshold_count)]
    reconstructed = global_frame[mode_columns].agg("/".join, axis=1)
    edge_boundaries = edge_frame.groupby("time", sort=False)["active_contest"].sum().reindex(global_frame.index)
    forbidden_tokens = ("future", "terminal", "outcome", "resolution")
    live_columns_clean = not any(token in column for column in lineage_frame for token in forbidden_tokens)

    completed_times = selected_scale["completed_event_information_time"]
    completed_time_valid = completed_times.isna() | completed_times.le(selected_scale["time"])
    current_times = selected_scale["current_event_information_time"]
    current_time_valid = current_times.isna() | current_times.le(selected_scale["time"])
    dcc_ids_valid = selected_scale.loc[selected_scale["dcc_flag"], "dcc_event_id"].notna().all()

    known_scale = selected_scale.loc[selected_scale["mode"].notna()]
    progress_valid = (
        known_scale["reversal_progress"].ge(-1e-12).all()
        and known_scale["reversal_progress"].lt(1.0 + 1e-10).all()
    )
    r_mask = known_scale["live_t"].gt(0.0) & known_scale["live_r"].notna()
    r_identity = np.allclose(
        known_scale.loc[r_mask, "live_r"],
        known_scale.loc[r_mask, "live_tmv"] * known_scale.loc[r_mask, "threshold"] / known_scale.loc[r_mask, "live_t"],
        rtol=1e-12,
        atol=1e-15,
    )

    if lineage_frame.empty or resolution_frame.empty:
        resolution_bars_excluded = True
    else:
        known_resolutions = resolution_frame.dropna(subset=["resolution_information_time"])
        overlap = lineage_frame.merge(
            known_resolutions[["lineage_id", "resolution_information_time"]],
            on="lineage_id",
            how="inner",
        )
        resolution_bars_excluded = ~overlap["time"].eq(overlap["resolution_information_time"]).any()

    checks = {
        "scale_rows_complete": len(selected_scale) == len(prices) * threshold_count,
        "configuration_reconstructs": reconstructed.eq(global_frame["configuration"]).all(),
        "edge_boundaries_reconstruct": edge_boundaries.eq(global_frame["direction_boundaries"]).all(),
        "feature_time_is_row_time": selected_scale["feature_information_time"].eq(selected_scale["time"]).all(),
        "completed_information_is_causal": completed_time_valid.all(),
        "current_event_information_is_causal": current_time_valid.all(),
        "dcc_event_ids_present": dcc_ids_valid,
        "reversal_progress_is_pre_threshold": progress_valid,
        "live_r_identity": r_identity,
        "live_lineage_has_no_outcome_columns": live_columns_clean,
        "resolution_bars_excluded_from_live_lineage": resolution_bars_excluded,
        "event_view_is_union_of_dcc_bars": view["event_state"]["dcc_count_this_bar"].gt(0).all(),
    }
    return pd.DataFrame({"threshold_count": threshold_count, "check": checks.keys(), "passed": checks.values()})


audit_parts = []
summary_records = []
full_view = None
for threshold_count in range(2, len(THRESHOLDS) + 1):
    view = build_representation_view(prices, scale_state, threshold_count)
    audit = audit_representation(scale_state, view, threshold_count)
    if not audit["passed"].all():
        failed = audit.loc[~audit["passed"], "check"].tolist()
        raise AssertionError(f"N={threshold_count} representation checks failed: {failed}")
    audit_parts.append(audit)

    global_frame = view["global_state"]
    resolutions = view["lineage_resolutions"]
    summary_records.append({
        "threshold_count": threshold_count,
        "maximum_threshold": THRESHOLDS[threshold_count - 1],
        "initialized_bars": len(global_frame),
        "theoretical_configurations": 2 ** threshold_count,
        "observed_configurations": global_frame["configuration_code"].nunique(),
        "broad_agreement_share": global_frame["broad_agreement"].mean(),
        "coherent_frontier_share": global_frame["coherent_frontier"].mean(),
        "fragmented_share": global_frame["hierarchy_shape"].eq(HierarchyShape.FRAGMENTED.value).mean(),
        "union_event_bars": len(view["event_state"]),
        "multi_scale_same_bar_share": global_frame["multi_scale_same_bar"].mean(),
        "broad_root_lineages": resolutions["lineage_id"].nunique() if not resolutions.empty else 0,
        "ambiguous_lineage_share": (
            resolutions["resolution"].eq(LineageResolution.AMBIGUOUS.value).mean()
            if not resolutions.empty else np.nan
        ),
    })

    if threshold_count == len(THRESHOLDS):
        full_view = view
    else:
        del view
        gc.collect()

representation_audit = pd.concat(audit_parts, ignore_index=True)
representation_summary = pd.DataFrame.from_records(summary_records).set_index("threshold_count")

In [10]:
display(representation_summary.style.format({
    "maximum_threshold": "{:.4%}",
    "initialized_bars": "{:,}",
    "theoretical_configurations": "{:,}",
    "observed_configurations": "{:,}",
    "broad_agreement_share": "{:.2%}",
    "coherent_frontier_share": "{:.2%}",
    "fragmented_share": "{:.2%}",
    "union_event_bars": "{:,}",
    "multi_scale_same_bar_share": "{:.3%}",
    "broad_root_lineages": "{:,}",
    "ambiguous_lineage_share": "{:.2%}",
}))
display(pd.crosstab(
    representation_audit["check"],
    representation_audit["threshold_count"],
    values=representation_audit["passed"],
    aggfunc="all",
).fillna(False))

,maximum_threshold,initialized_bars,theoretical_configurations,observed_configurations,broad_agreement_share,coherent_frontier_share,fragmented_share,union_event_bars,multi_scale_same_bar_share,broad_root_lineages,ambiguous_lineage_share
threshold_count,,,,,,,,,,,
2,0.3000%,"258,537",4,4,72.15%,100.00%,0.00%,"4,010",0.029%,"2,043",3.72%
3,0.4500%,"258,332",8,8,51.61%,93.22%,6.78%,"4,645",0.039%,"1,518",5.20%
4,0.6750%,"258,301",16,16,36.55%,82.26%,17.74%,"4,935",0.043%,"1,106",5.97%
5,1.0125%,"257,813",32,32,26.39%,70.63%,29.37%,"5,062",0.045%,799,5.88%


threshold_count,2,3,4,5
check,,,,
completed_information_is_causal,True,True,True,True
configuration_reconstructs,True,True,True,True
current_event_information_is_causal,True,True,True,True
dcc_event_ids_present,True,True,True,True
edge_boundaries_reconstruct,True,True,True,True
event_view_is_union_of_dcc_bars,True,True,True,True
feature_time_is_row_time,True,True,True,True
live_lineage_has_no_outcome_columns,True,True,True,True
live_r_identity,True,True,True,True


## Inspect the full five-threshold representation

The five-threshold view is now inspected as three causal tables plus a separate diagnostic resolution table. The summaries below describe representation occupancy; they are not propagation forecasts.

In [11]:
global_state = full_view["global_state"]
edge_state = full_view["edge_state"]
event_state = full_view["event_state"]
lineage_state = full_view["lineage_state"]
lineage_resolutions = full_view["lineage_resolutions"]

scale_summary = scale_state.groupby(["threshold_index", "threshold"], sort=True).agg(
    events=("dcc_flag", "sum"),
    initialized_bars=("mode", "count"),
    median_mode_age=("mode_age_bars", "median"),
    median_live_tmv=("live_tmv", "median"),
    median_live_t=("live_t", "median"),
    median_live_r=("live_r", "median"),
    median_reversal_progress=("reversal_progress", "median"),
)
edge_summary_records = []
for (lower_index, upper_index), group in edge_state.groupby(
    ["lower_threshold_index", "upper_threshold_index"], sort=True
):
    active = group.loc[group["active_contest"]]
    edge_summary_records.append({
        "edge": f"T{lower_index + 1:02d}->T{upper_index + 1:02d}",
        "threshold_ratio": group["threshold_ratio"].iloc[0],
        "disagreement_share": group["active_contest"].mean(),
        "same_bar_transition_share": group["same_bar_transition"].mean(),
        "median_active_age": active["contest_age_bars"].median(),
        "median_target_progress": active["target_progress"].median(),
        "p90_target_progress": active["target_progress"].quantile(0.90),
    })
edge_summary = pd.DataFrame.from_records(edge_summary_records).set_index("edge")

display(scale_summary.style.format({
    "events": "{:,}",
    "initialized_bars": "{:,}",
    "median_mode_age": "{:.1f}",
    "median_live_tmv": "{:.3f}",
    "median_live_t": "{:.1f}",
    "median_live_r": "{:.3e}",
    "median_reversal_progress": "{:.3f}",
}))
display(edge_summary.style.format({
    "threshold_ratio": "{:.3f}",
    "disagreement_share": "{:.2%}",
    "same_bar_transition_share": "{:.3%}",
    "median_active_age": "{:.1f}",
    "median_target_progress": "{:.3f}",
    "p90_target_progress": "{:.3f}",
}))

,,events,initialized_bars,median_mode_age,median_live_tmv,median_live_t,median_live_r,median_reversal_progress
threshold_index,threshold,,,,,,,
0,0.002000,"2,697","258,546",68.0,1.486,109.0,2.870e-05,0.279
1,0.003000,"1,389","258,537",128.0,1.426,204.0,2.158e-05,0.291
2,0.004500,683,"258,332",241.0,1.392,407.0,1.632e-05,0.299
3,0.006750,308,"258,301",545.0,1.475,859.0,1.139e-05,0.305
4,0.010125,146,"257,813",1076.0,1.426,1705.0,8.442e-06,0.290


,threshold_ratio,disagreement_share,same_bar_transition_share,median_active_age,median_target_progress,p90_target_progress
edge,,,,,,
T01->T02,1.500,27.86%,0.029%,30.0,0.638,0.855
T02->T03,1.500,27.32%,0.015%,56.0,0.638,0.850
T03->T04,1.500,28.10%,0.003%,118.0,0.611,0.836
T04->T05,1.500,26.25%,0.001%,205.0,0.619,0.830


## Bar-time versus union-event-time

Bar time weights persistent structures by their duration. Union-event time samples only bars on which at least one threshold generated a DCC and therefore weights structural change. Neither is universally correct; the observation clock must be chosen explicitly before fitting a classifier.

In [12]:
bar_shape_share = global_state["hierarchy_shape"].value_counts(normalize=True).rename("bar_time")
event_shape_share = event_state["hierarchy_shape"].value_counts(normalize=True).rename("union_event_time")
clock_comparison = pd.concat([bar_shape_share, event_shape_share], axis=1).fillna(0.0)
display(clock_comparison.style.format("{:.2%}"))

clock_figure = go.Figure()
for column, color in [("bar_time", "#2563eb"), ("union_event_time", "#f97316")]:
    clock_figure.add_bar(x=clock_comparison.index, y=clock_comparison[column], name=column, marker_color=color)
clock_figure.update_layout(
    title=f"{SYMBOL}: hierarchy-shape occupancy depends on the observation clock",
    template="plotly_white",
    barmode="group",
    height=480,
)
clock_figure.update_yaxes(title="Share", tickformat=".0%")
clock_figure.show()

,bar_time,union_event_time
hierarchy_shape,,
coherent_frontier,44.24%,45.95%
fragmented,29.37%,38.29%
broad_agreement,26.39%,15.76%


## Hand-worked hierarchy examples

These examples verify that breadth and fragmentation retain scale topology. A one-boundary configuration is a coherent propagation frontier. Two or more boundaries mean that no single propagation-depth label can faithfully describe the hierarchy.

In [13]:
def summarize_mode_vector(name: str, modes: tuple[str, ...]) -> dict[str, object]:
    signs = np.array([MODE_SIGN[mode] for mode in modes], dtype=np.int8)
    boundaries = signs[:-1] != signs[1:]
    boundary_count = int(boundaries.sum())
    small_breadth = int(np.argmax(boundaries) + 1) if boundary_count else len(modes)
    outer_breadth = int(np.argmax(boundaries[::-1]) + 1) if boundary_count else len(modes)
    if boundary_count == 0:
        shape = HierarchyShape.BROAD_AGREEMENT.value
        counter_depth = 0
    elif boundary_count == 1:
        shape = HierarchyShape.COHERENT_FRONTIER.value
        counter_depth = small_breadth
    else:
        shape = HierarchyShape.FRAGMENTED.value
        counter_depth = pd.NA
    return {
        "example": name,
        "configuration": "/".join(modes),
        "boundary_vector": "/".join(boundaries.astype(np.int8).astype(str)),
        "direction_boundaries": boundary_count,
        "hierarchy_shape": shape,
        "small_contiguous_breadth": small_breadth,
        "outer_contiguous_breadth": outer_breadth,
        "counter_scale_depth": counter_depth,
    }


worked_examples = pd.DataFrame.from_records([
    summarize_mode_vector("broad up", ("up", "up", "up", "up", "up")),
    summarize_mode_vector("small pullback", ("down", "up", "up", "up", "up")),
    summarize_mode_vector("reached medium", ("down", "down", "up", "up", "up")),
    summarize_mode_vector("reached macro", ("down", "down", "down", "down", "up")),
    summarize_mode_vector("fragmented", ("down", "up", "down", "up", "up")),
]).set_index("example")
display(worked_examples)

,configuration,boundary_vector,direction_boundaries,hierarchy_shape,small_contiguous_breadth,outer_contiguous_breadth,counter_scale_depth
example,,,,,,,
broad up,up/up/up/up/up,0/0/0/0,0,broad_agreement,5,5,0
small pullback,down/up/up/up/up,1/0/0/0,1,coherent_frontier,1,4,1
reached medium,down/down/up/up/up,0/1/0/0,1,coherent_frontier,2,3,2
reached macro,down/down/down/down/up,0/0/0/1,1,coherent_frontier,4,1,4
fragmented,down/up/down/up/up,1/1/1/0,3,fragmented,1,2,<NA>


## Economic casebook on actual bars

The mode heatmap is ordered from the smallest threshold at the top to the outer threshold at the bottom. DCC markers on price use one color per threshold. The state shown at every timestamp is the post-close causal state; simultaneous markers reveal bars whose intrabar threshold order cannot be recovered.

In [14]:
def plot_state_window(
    price_series: pd.Series, scale_frame: pd.DataFrame, global_frame: pd.DataFrame, end_time: pd.Timestamp,
    threshold_count: int, bars: int, title: str,
) -> go.Figure:
    end_position = int(global_frame.loc[end_time, "bar_position"])
    start_position = max(0, end_position - bars + 1)
    window_times = price_series.index[start_position:end_position + 1]
    window_prices = price_series.loc[window_times]
    figure = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.05)
    figure.add_trace(go.Scatter(
        x=window_times, y=window_prices, mode="lines", name="Close", line={"color": "#64748b", "width": 1.0},
    ), row=1, col=1)

    selected = scale_frame.loc[
        scale_frame["threshold_index"].lt(threshold_count)
        & scale_frame["time"].ge(window_times[0])
        & scale_frame["time"].le(window_times[-1])
    ]
    for threshold_index in range(threshold_count):
        events = selected.loc[selected["threshold_index"].eq(threshold_index) & selected["dcc_flag"]]
        figure.add_trace(go.Scatter(
            x=events["time"],
            y=events["price"],
            mode="markers",
            name=f"T{threshold_index + 1:02d} {THRESHOLDS[threshold_index]:.3%} DCC",
            marker={"size": 5, "color": PLOT_COLORS[threshold_index], "symbol": "circle"},
        ), row=1, col=1)

    mode_matrix = scale_matrix(scale_frame, "mode_sign", threshold_count).loc[window_times].T
    y_labels = [f"T{index + 1:02d} {THRESHOLDS[index]:.3%}" for index in range(threshold_count)]
    figure.add_trace(go.Heatmap(
        x=window_times,
        y=y_labels,
        z=mode_matrix.to_numpy(dtype=float),
        zmin=-1,
        zmax=1,
        colorscale=[[0.0, "#dc2626"], [0.499, "#dc2626"], [0.5, "#16a34a"], [1.0, "#16a34a"]],
        colorbar={"title": "Mode", "tickvals": [-1, 1], "ticktext": ["down", "up"]},
        showscale=True,
    ), row=2, col=1)
    figure.add_vline(x=end_time, line_width=1.2, line_dash="dash", line_color="#111827")
    figure.update_layout(title=title, template="plotly_white", height=700, hovermode="x unified")
    figure.update_yaxes(title="Close", row=1, col=1)
    return figure


case_records = []
for shape in HierarchyShape:
    candidates = global_state.loc[global_state["hierarchy_shape"].eq(shape.value)]
    if candidates.empty:
        continue
    case = candidates.iloc[-1]
    case_records.append(case[[
        "time", "configuration", "hierarchy_shape", "direction_boundaries",
        "small_contiguous_breadth", "outer_contiguous_breadth", "max_reversal_progress",
    ]].to_dict())
    plot_state_window(
        prices,
        scale_state,
        global_state,
        case.name,
        len(THRESHOLDS),
        CASE_WINDOW_BARS,
        f"{SYMBOL}: recent {shape.value} example ending {case.name}",
    ).show()

casebook = pd.DataFrame.from_records(case_records).set_index("hierarchy_shape")
display(casebook)

,time,configuration,direction_boundaries,small_contiguous_breadth,outer_contiguous_breadth,max_reversal_progress
hierarchy_shape,,,,,,
broad_agreement,2026-11-09 03:10:00+02:00,down/down/down/down/down,0,5,5,0.974294
coherent_frontier,2026-11-09 13:30:00+02:00,up/up/up/down/down,1,3,2,0.853825
fragmented,2026-11-05 21:10:00+02:00,up/down/up/up/up,2,1,3,0.812031


## Live lineage context versus diagnostic resolutions

The first table contains only information known on each live row. The second table is explicitly outcome-conditioned and diagnostic. Do not merge the resolution fields backward onto the live rows when building a classifier.

In [15]:
if lineage_state.empty:
    print("No clean live broad-root lineage rows were found.")
else:
    recent_lineage = lineage_state.tail(12).merge(
        global_state[["time", "configuration", "hierarchy_shape", "direction_boundaries"]].reset_index(drop=True),
        on="time",
        how="left",
        validate="many_to_one",
    )
    recent_lineage = recent_lineage.merge(
        edge_state[[
            "time", "lower_threshold_index", "target_progress", "max_target_progress",
            "progress_drawdown", "contest_age_bars",
        ]],
        left_on=["time", "active_lower_threshold_index"],
        right_on=["time", "lower_threshold_index"],
        how="left",
        validate="many_to_one",
    )
    display(recent_lineage.drop(columns="lower_threshold_index"))

resolution_summary = pd.crosstab(
    lineage_resolutions["resolution"],
    lineage_resolutions["attained_depth"],
    margins=True,
)
display(resolution_summary)
print("Resolution summary is diagnostic/hindsight; lineage_state is the causal model-input candidate.")

,lineage_id,seed_time,stage_start_event_id,stage_start_time,time,feature_information_time,bar_position,target_mode,attained_depth,active_lower_threshold_index,...,lineage_age_bars,stage_age_bars,timing_clean,configuration,hierarchy_shape,direction_boundaries,target_progress,max_target_progress,progress_drawdown,contest_age_bars
0,T01-E02697,2026-11-09 03:15:00+02:00,T03-E00683,2026-11-09 10:15:00+02:00,2026-11-09 12:35:00+02:00,2026-11-09 12:35:00+02:00,258561,up,2,2,...,112,28,True,up/up/up/down/down,coherent_frontier,1,0.841612,0.867149,0.025537,28.0
1,T01-E02697,2026-11-09 03:15:00+02:00,T03-E00683,2026-11-09 10:15:00+02:00,2026-11-09 12:40:00+02:00,2026-11-09 12:40:00+02:00,258562,up,2,2,...,113,29,True,up/up/up/down/down,coherent_frontier,1,0.877142,0.877142,0.000000,29.0
2,T01-E02697,2026-11-09 03:15:00+02:00,T03-E00683,2026-11-09 10:15:00+02:00,2026-11-09 12:45:00+02:00,2026-11-09 12:45:00+02:00,258563,up,2,2,...,114,30,True,up/up/up/down/down,coherent_frontier,1,0.852715,0.877142,0.024427,30.0
3,T01-E02697,2026-11-09 03:15:00+02:00,T03-E00683,2026-11-09 10:15:00+02:00,2026-11-09 12:50:00+02:00,2026-11-09 12:50:00+02:00,258564,up,2,2,...,115,31,True,up/up/up/down/down,coherent_frontier,1,0.840502,0.877142,0.036640,31.0
4,T01-E02697,2026-11-09 03:15:00+02:00,T03-E00683,2026-11-09 10:15:00+02:00,2026-11-09 12:55:00+02:00,2026-11-09 12:55:00+02:00,258565,up,2,2,...,116,32,True,up/up/up/down/down,coherent_frontier,1,0.847164,0.877142,0.029978,32.0
5,T01-E02697,2026-11-09 03:15:00+02:00,T03-E00683,2026-11-09 10:15:00+02:00,2026-11-09 13:00:00+02:00,2026-11-09 13:00:00+02:00,258566,up,2,2,...,117,33,True,up/up/up/down/down,coherent_frontier,1,0.852715,0.877142,0.024427,33.0
6,T01-E02697,2026-11-09 03:15:00+02:00,T03-E00683,2026-11-09 10:15:00+02:00,2026-11-09 13:05:00+02:00,2026-11-09 13:05:00+02:00,258567,up,2,2,...,118,34,True,up/up/up/down/down,coherent_frontier,1,0.867149,0.877142,0.009993,34.0
7,T01-E02697,2026-11-09 03:15:00+02:00,T03-E00683,2026-11-09 10:15:00+02:00,2026-11-09 13:10:00+02:00,2026-11-09 13:10:00+02:00,258568,up,2,2,...,119,35,True,up/up/up/down/down,coherent_frontier,1,0.880473,0.880473,0.000000,35.0
8,T01-E02697,2026-11-09 03:15:00+02:00,T03-E00683,2026-11-09 10:15:00+02:00,2026-11-09 13:15:00+02:00,2026-11-09 13:15:00+02:00,258569,up,2,2,...,120,36,True,up/up/up/down/down,coherent_frontier,1,0.886024,0.886024,0.000000,36.0
9,T01-E02697,2026-11-09 03:15:00+02:00,T03-E00683,2026-11-09 10:15:00+02:00,2026-11-09 13:20:00+02:00,2026-11-09 13:20:00+02:00,258570,up,2,2,...,121,37,True,up/up/up/down/down,coherent_frontier,1,0.844943,0.886024,0.041081,37.0


attained_depth,0,1,2,3,4,All
resolution,,,,,,
ambiguous_same_bar,32,9,4,2,0,47
reached_outer,0,0,0,0,126,126
rejected_at_frontier,262,172,121,70,0,625
unresolved_at_sample_end,0,0,1,0,0,1
All,294,181,126,72,126,799


Resolution summary is diagnostic/hindsight; lineage_state is the causal model-input candidate.


## Representation interpretation lock

A causal read at one bar should now sound like:

> Small and medium agree downward against three larger upward scales. There is one coherent boundary between medium and large. The broad-root lineage has attained medium and is currently contesting large. Its terminal depth is unknown.

The representation separates three facts that must not be collapsed:

- **snapshot topology:** what all modes and boundaries look like now;
- **live dynamics:** ages, TMV/T/R, progress, drawdown, and firing density now;
- **lineage history:** what depth has already been causally attained by a pinned seed.

Two configurations with identical up/down counts can have different scale topology. Two lineages with the same snapshot can have different attained histories. A future classifier should receive enough information to distinguish both without receiving the future resolution.

# Multiscale representation completion gate

Before fitting any regime classifier, be able to explain and verify:

1. Why `scale_state` is the canonical arbitrary-N node representation.
2. Why there are exactly $N-1$ adjacent edge states for $N$ thresholds.
3. Why the complete ordered mode vector must be retained even when breadth and fragmentation are available.
4. Why threshold values and adjacent ratios are part of the representation contract.
5. The difference between current-leg TMV/T/R and the previous completed-leg TMV/T/R.
6. The difference between leg excursion and reversal progress.
7. Why an adjacent disagreement is a current state, while propagation or rejection is a later outcome.
8. Why attained lineage depth is causal but terminal depth is not.
9. Why the resolution bar is absent from its resolving live stage.
10. Why same-bar multiscale transitions are marked ambiguous under close-only observation.
11. Why bar-time and union-event-time produce different state populations.
12. Why no full-sample normalization belongs in a tradeable representation.
13. Why fragmented configurations cannot be faithfully reduced to one propagation-depth number.
14. Why the same construction and identity checks must pass for 2, 3, 4, and 5 thresholds.
15. Why `lineage_resolutions` is diagnostic and must never be joined backward into `lineage_state`.

Once these gates are locked in, the next phase is representation evaluation and regime classification: begin with interpretable structural rules, then compare clustering and causal-filtered probabilistic models. Every model must demonstrate an addition beyond the directly observable hierarchy.